In [ ]:
import os
import random
import shutil
from astropy.io import fits

# Definir los directorios de origen y destino
src_folder = "spectrums"
dst_folder = "spectrums training 100k"

# Crear la carpeta de destino si no existe
if not os.path.exists(dst_folder):
    os.makedirs(dst_folder)

# Definir los rangos de redshift y los mínimos requeridos para cada uno
bin_ranges = [(0, 0.1), (0.1, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8)]
min_required_values = {
    (0, 0.1): 20000,
    (0.1, 1): 15000,
    (1, 2): 20000,
    (2, 3): 15000,
    (3, 4): 5000,
    (4, 5): 2000,
    (5, 6): 900,
    (6, 7): 850,
    (7, 8): 240
}

# Inicializar contadores y diccionario para almacenar los archivos copiados por rango
selected_counts = {br: 0 for br in bin_ranges}
selected_files_by_bin = {br: [] for br in bin_ranges}

# Obtener la lista de archivos de la carpeta origen y barajarla aleatoriamente
all_files = [f for f in os.listdir(src_folder) if f.lower().endswith(".fits")]
random.shuffle(all_files)

# Iterar sobre los archivos de forma aleatoria
for file in all_files:
    src_path = os.path.join(src_folder, file)
    try:
        with fits.open(src_path) as hdul:
            redshift = hdul[2].data["Z"][0]
    except Exception as e:
        print(f"Error procesando {file}: {e}")
        continue

    # Verificar en qué rango se ubica el archivo
    for br in bin_ranges:
        lower, upper = br
        if lower <= redshift < upper:
            # Copiar el archivo si no se ha alcanzado el mínimo en ese rango
            if selected_counts[br] < min_required_values[br]:
                dst_path = os.path.join(dst_folder, file)
                shutil.copy(src_path, dst_path)
                selected_counts[br] += 1
                selected_files_by_bin[br].append(file)
                print(f"Copiado: {file} para rango {lower}-{upper} ({selected_counts[br]}/{min_required_values[br]})")
            break

    # Si ya se han cumplido los mínimos en todos los rangos, se interrumpe el bucle
    if all(selected_counts[br] >= min_required_values[br] for br in bin_ranges):
        print("Se han alcanzado los mínimos requeridos para todos los rangos.")
        break

# Contar los archivos que hay en la carpeta destino
dst_files = [f for f in os.listdir(dst_folder) if f.lower().endswith(".fits")]
num_dst = len(dst_files)
print(f"Total de archivos en destino tras cumplir mínimos: {num_dst}")

# Si faltan archivos para llegar a 100k, se añaden archivos aleatorios
if num_dst < 100000:
    needed = 100000 - num_dst
    print(f"Faltan {needed} archivos para llegar a 100k. Se añadirán archivos aleatorios.")
    # Excluir los archivos que ya han sido copiados
    already_copied = set(dst_files)
    remaining_files = [f for f in os.listdir(src_folder) if f.lower().endswith(".fits") and f not in already_copied]
    # Mezclar aleatoriamente los archivos restantes
    random.shuffle(remaining_files)
    additional_files = remaining_files[:needed]
    for file in additional_files:
        src_path = os.path.join(src_folder, file)
        dst_path = os.path.join(dst_folder, file)
        shutil.copy(src_path, dst_path)
        print(f"Copiado archivo adicional: {file}")
else:
    print("La carpeta destino ya tiene 100k o más archivos.")

# Verificar el conteo final en la carpeta destino
final_files = [f for f in os.listdir(dst_folder) if f.lower().endswith(".fits")]
print(f"Conteo final en destino: {len(final_files)} archivos")

In [ ]:
import os
import numpy as np
import pickle
from astropy.io import fits

# Directorio de destino (donde se encuentran los archivos ya copiados)
dst_folder = "spectrums training 100k"

# Lista para almacenar los valores de redshift
redshift_values = []

# Recorrer todos los archivos .fits en la carpeta
for filename in os.listdir(dst_folder):
    if filename.lower().endswith('.fits'):
        file_path = os.path.join(dst_folder, filename)
        try:
            with fits.open(file_path) as hdul:
                # Extraer redshift (se asume que está en la extensión 2, campo "Z")
                redshift = hdul[2].data["Z"][0]
                redshift_values.append(redshift)
        except Exception as e:
            print(f"Error procesando {filename}: {e}")

# Convertir la lista a un arreglo de numpy
redshift_array = np.array(redshift_values)

# Guardar el array de redshifts en un archivo pickle
with open("extra/redshift_array.pkl", "wb") as f:
    pickle.dump(redshift_array, f)

print(f"Guardado redshift_array con {redshift_array.size} elementos en 'redshift_array.pkl'.")

# Opcional: mostrar el histograma original
import matplotlib.pyplot as plt
bins = np.arange(0, redshift_array.max() + 0.1, 0.1)
plt.figure(figsize=(18,16))
plt.hist(redshift_array, bins=bins, color='skyblue', edgecolor='black')
plt.xlabel("Redshift")
plt.ylabel("Número de archivos")
plt.title("Histograma de redshifts (bin width = 0.1)")
plt.xlim(0, 7.1)
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# Directorio que contiene los archivos FITS
dst_folder = "spectrums training 100k"

plt.figure(figsize=(18,16))
# Dibujar todos los puntos: cada punto se dibuja en (redshift, redshift)
plt.scatter(redshift_array, redshift_array, color='blue', s=5, alpha=0.5, label='Puntos de redshift')

# Dibujar la función y = x
x_vals = np.linspace(0, redshift_array.max(), 100)
plt.plot(x_vals, x_vals, color='red', linestyle='--', label='y = x')

plt.xlabel("Redshift")
plt.ylabel("Redshift")
plt.title("Representación de puntos de redshift y la función y = x")
plt.legend()
plt.show()